In [ ]:
%pip install diffusers transformers accelerate torch torchvision Pillow peft

# 1. Training

## Imports

In [ ]:
import os
import csv
from pathlib import Path
from PIL import Image
from typing import List

import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# For prompt: convert text to numbers, convert number to embeddings.
#             UNet will use the embeddings to guide image creation.
from transformers import CLIPTokenizer, CLIPTextModel

# AutoencoderKL (compress & reconstruct images): for compression and decompression btw pixel space and latent space.
# UNet2DConditionModel (predicts the noise): U-Net NN that takes (a noisy latent, timestep and text embedding) and predicts the noise to be removed.
# DDPMScheduler (adds noise - training): handles the noise process. How much noise to add at a certain timestemp. add_noise() - to use during training.
# UniPCMultistepScheduler (removes noise - inference): the denoising solver - to be used during inference.
# StableDiffusionPipeline (wraps the above together): wraper for the 5 components (tokenizer, text encoder, VAE, UNet, scheduler) - to use at inference.

# Training:
#     Image -> [AutoencoderKL] ->
#     Latent -> [DDPMScheduler (add noise)] ->
#     Noisy latent -> [UNet2DConditionModel] ->
#     Predicted noise -> [loss]

# Inference:
#     Random noise -> [UniPCMultistepScheduler] ->
#     Iterative denoising -> [UNet2DConditionModel] ->
#     Clean latent -> [AutoencoderKL] ->
#     Final Image
from diffusers import AutoencoderKL, DDPMScheduler, StableDiffusionPipeline, UNet2DConditionModel, UniPCMultistepScheduler
from diffusers import StableDiffusionImg2ImgPipeline
from diffusers.utils import load_image, make_image_grid

# LoraConfig: defines how LORA layers are inserted in the model.
# get_peft_model: takes the pretrained model and injects the Lora adapters.

# The original U-Net weights are frozen ->
# LoRA layers are added into attention ->
# the optimizer updates only the LoRA and does not retrain the millions of parameters
from peft import LoraConfig, get_peft_model

## Dataset

In [ ]:
class TrainDataset(Dataset):
    """
    Dataset class for training data

    """
    def __init__(self, folder_path: str):
        self.dataset_dir = Path(folder_path)
        #print("dataset init ", self.dataset_dir)
        self.images_paths = sorted(self.dataset_dir.glob("*.jpg"))
        print("number of paths",len(self.images_paths))
        #print(self.images_paths[:10])
        self.transform = self.get_transform()

    def __len__(self) -> int:
        return len(self.images_paths)

    def __getitem__(self, index: int) -> torch.Tensor:
        img_path =  self.images_paths[index]
        img = Image.open(img_path).convert("RGB")
        img_tensor = self.transform(img)
        return img_tensor

    @staticmethod
    def get_transform(
        img_size: int = 512,
        norm_scalar: List[float] = [0.5]
    ) -> transforms.Compose:
        t = transforms.Compose([
            transforms.Resize(img_size, interpolation=transforms.InterpolationMode.LANCZOS),
            transforms.CenterCrop(img_size),
            transforms.ToTensor(),
            transforms.Normalize(norm_scalar, norm_scalar)
        ])
        return t

## Configurations

In [ ]:
STYLE_IMAGES_PATH = "vangogh"
MODELS_OUTPUT_DIR = "models"



In [ ]:
os.makedirs(MODELS_OUTPUT_DIR, exist_ok=True)

In [ ]:
MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"

In [ ]:
STYLE_TYPE = "van Gogh"
PROMP_INPUT = f"stylization in {STYLE_TYPE} style"

In [ ]:
LORA_RANK = 32 # 4-8: style is subtle, 16-32: style is stronger
LORA_ALPHA = 32 # scaling factor. Should be the same as rank
LORA_DROPOUT = 0.1 # make lora a little more noisy during training. This will allow to learn a cleaner style.

In [ ]:
TRAINABLE_LAYERS = ["to_q", "to_k", "to_v", "to_out.0"]
EPOCHS = 20
TRAINING_STRENGTH = 0.5 # range btw 0.3 - 0.5 for best results (brushstrokes, colors, texture)
LEARNING_RATE = 1e-5
BATCH_SIZE = 1
NUMBER_OF_WORKERS = 2
GRADIENT_ACCUMULATION = 8
SEED = 42


## Device & data type

In [ ]:
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Device: {device}; dtype: {dtype}")

## LoRA & Models

In [ ]:
tokenizer = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
prompt_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder", torch_dtype=dtype).to(device)

#AutoencoderKL, DDPMScheduler, StableDiffusionPipeline, UNet2DConditionModel, UniPCMultistepScheduler
autoencoder = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=dtype).to(device)
train_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet").to(device)

In [ ]:
# Freeze
prompt_encoder.requires_grad_(False)
autoencoder.requires_grad_(False)
unet.requires_grad_(False)

In [ ]:
# Turn the normal stable diffusion unet to LoRA unet. Only some adapter layers are trainable
lora_configurations = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    init_lora_weights="gaussian",
    target_modules=TRAINABLE_LAYERS,
    bias="none"
)

lora_unet = get_peft_model(unet, lora_configurations) # inject lora weights and freezes original unet

lora_parameters = [param for param in lora_unet.parameters() if param.requires_grad]
optimizer = optim.AdamW(lora_parameters, lr=LEARNING_RATE)

## Training process

In [ ]:
dataset = TrainDataset(STYLE_IMAGES_PATH)
data_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=NUMBER_OF_WORKERS,
    pin_memory=torch.cuda.is_available()
)
prompts = [PROMP_INPUT] * BATCH_SIZE
timestep_max = int(train_scheduler.config.num_train_timesteps * TRAINING_STRENGTH)


In [ ]:
# set unet into training mode
lora_unet.train()

In [ ]:
step_counter = 0
loss_log = []
for epoch in range(EPOCHS):
    #print(lora_parameters[0].grad)
    print(f"Epoch {epoch + 1}/{EPOCHS}:")
    for batch in data_loader:
        img_tensor = batch.to(device, dtype=dtype)

        with torch.no_grad():
            # encodes the tensor to latent space
            latent_img = autoencoder.encode(img_tensor).latent_dist.sample() * autoencoder.config.scaling_factor

            prompt_tokens = tokenizer(
                prompts,
                truncation=True,
                padding='max_length',
                max_length=tokenizer.model_max_length,
                return_tensors='pt'
            ).input_ids.to(device)
            promp_embeddings = prompt_encoder(prompt_tokens)[0]

        # adding noise to the latent space img.
        noise = torch.randn_like(latent_img)
        timestep = torch.randint(0, timestep_max, (BATCH_SIZE,), device=device, dtype=torch.long)
        noisy_latent_img = train_scheduler.add_noise(latent_img, noise, timestep)


        predicted_noise = lora_unet(noisy_latent_img.float(), timestep, encoder_hidden_states=promp_embeddings.float()).sample

        loss = F.mse_loss(predicted_noise.float(), noise.float()) / GRADIENT_ACCUMULATION
        loss.backward()

        if step_counter % GRADIENT_ACCUMULATION == 0:
            torch.nn.utils.clip_grad_norm_(lora_parameters, 1.0)
            optimizer.step()
            optimizer.zero_grad()

        step_counter+=1
        if step_counter % 10 == 0:
            current_loss = loss.item() * GRADIENT_ACCUMULATION
            loss_log.append({"iteration": step_counter, "loss": current_loss})
            print(f"Step: {step_counter}, loss: {current_loss}")
        #if step_counter % 500 == 0:
            model_name = f"model_{step_counter}"
            model_dir = os.path.join(MODELS_OUTPUT_DIR, model_name)
            lora_unet.save_pretrained(model_dir)

In [ ]:
model_name = "final"
model_dir = os.path.join(MODELS_OUTPUT_DIR, model_name)
lora_unet.save_pretrained(model_dir)